# 7. Control Loop Start (Expanded)

Starts the periodic control loop thread once initial state is received.

---

```python id="9t3s0w"
def Start(self):
    while not self.first_update:
        time.sleep(0.5)

    self.thread = RecurrentThread(
        interval=self.dt,
        target=self.ControlLoop,
        name="arm_control"
    )
    self.thread.Start()
```

---

## 🧠 Big Picture: What This Function Does

This function does two critical things:

1. **Waits until the robot state is available**
2. **Starts a real-time control loop running in its own thread**

---

## ⏳ Step 1: Wait for Valid State

```python id="p9p9k2"
while not self.first_update:
    time.sleep(0.5)
```

---

### Why is this necessary?

Recall from Section 5:

```python
self.first_update = True
```

is only set after receiving **real robot data**.

---

### Without this loop:

Your program might start the control loop with:

```python
self.low_state = None
```

Which would lead to:

* ❌ Invalid reads
* ❌ Crashes
* ❌ Dangerous robot motion

---

### What this loop ensures:

```text
Do not move the robot until we KNOW its current state
```

---

### 🧠 Engineering Insight

This is a **synchronization barrier** between:

```text
Perception (state input) → Control (action output)
```

---

### ⚠️ Why `time.sleep(0.5)`?

* Prevents CPU spinning (busy-waiting)
* Reduces unnecessary load

---

### Tradeoff:

| Value              | Effect                    |
| ------------------ | ------------------------- |
| Small (e.g., 0.01) | Faster startup, more CPU  |
| Large (0.5)        | Slower startup, efficient |

---

## 🧵 Step 2: Create a Real-Time Thread

```python id="dnz7ec"
self.thread = RecurrentThread(
    interval=self.dt,
    target=self.ControlLoop,
    name="arm_control"
)
```

---

### What is `RecurrentThread`?

It is a helper that runs:

```text
ControlLoop() every dt seconds
```

---

### Equivalent concept:

```python
while True:
    ControlLoop()
    sleep(dt)
```

But implemented:

* More reliably
* With better timing guarantees

---

## ⏱️ Control Frequency

```python
interval = self.dt = 0.02
```

---

### This gives:

```text
1 / 0.02 = 50 Hz
```

So your control loop runs:

> 🔄 **50 times per second**

---

### Why this matters:

Robotics requires:

* Continuous command updates
* Smooth motion
* Stable feedback control

---

### ⚠️ If you stop publishing:

```text
Robot may:
- Freeze
- Relax joints
- Switch to fallback mode
```

---

## 🔁 Step 3: Define the Target Function

```python id="q9q60n"
target=self.ControlLoop
```

---

### This means:

Every 20 ms:

```text
→ Call ControlLoop()
```

---

### ControlLoop is where:

* Commands are computed
* Stages are updated
* Motion is generated

---

## 🧵 Step 4: Thread Naming

```python id="7nlx1s"
name="arm_control"
```

---

### Why name the thread?

* Debugging
* Logging
* Profiling

In complex systems:

```text
Multiple threads → need identification
```

---

## ▶️ Step 5: Start the Thread

```python id="9ndg7p"
self.thread.Start()
```

---

### What happens now?

```text
A new thread begins execution:
    → runs ControlLoop() repeatedly
```

---

### Important:

This does **NOT block** the main program.

---

## 🧠 Concurrency Model (CRITICAL)

After this function runs, your system now has:

```text
Thread 1 → ControlLoop (actions)
Thread 2 → LowStateHandler (state updates)
Main thread → idle / monitoring
```

---

### This is a **multi-threaded system**

---

## ⚠️ Key Implication

```text
State and control run in parallel
```

---

### Example:

* Thread A reads `self.low_state`
* Thread B updates `self.low_state`

---

### This is called:

> **Asynchronous execution**

---

## 🔄 Control System View

After `Start()` runs, your system becomes:

```text
Loop:
    Read state (from callback)
    Compute action (ControlLoop)
    Send command (publisher)
```

---

## 🤖 RL Interpretation

This is where your program becomes an **environment loop**:

```text
while True:
    state  ← LowStateHandler
    action ← ControlLoop
```

---

### Equivalent RL abstraction:

```python
while not done:
    action = policy(state)
    state  = env.step(action)
```

---

## 🔥 Critical Insight

Before this function:

```text
Your program is passive
```

After this function:

```text
Your program is actively controlling a robot in real time
```

---

## ⚙️ Failure Modes (Important for Teaching)

### 1. If `first_update` never becomes True

```text
→ Infinite wait
```

Cause:

* No DDS connection
* Wrong network interface

---

### 2. If thread doesn’t start

```text
→ Robot never moves
```

---

### 3. If `dt` is too large

```text
→ Slow, jerky motion
```

---

### 4. If `dt` is too small

```text
→ CPU overload
→ Network congestion
```

---

## 🧠 Teaching Insight

This is a great place to emphasize:

> “Robotics is inherently concurrent.”

Students should understand:

* Control is not sequential
* It is **continuous and parallel**

---

## 🔄 Analogy

Think of this like:

```text
Starting a heartbeat
```

Before:

* System is idle

After:

* System pulses at 50 Hz
* Continuously alive

---

## 🚀 Summary

This function:

| Step            | Role                        |
| --------------- | --------------------------- |
| Wait loop       | Ensures safe startup        |
| Thread creation | Defines control loop        |
| Interval (`dt`) | Sets control frequency      |
| Start()         | Activates real-time control |

---

> 🔥 This is the moment your program becomes a **real-time robot controller**.

